# Cybersec OpenEnv: GRPO training and evaluation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LonelyGuy-SE1/cybersec_env/blob/main/notebooks/cybersec_grpo.ipynb)

**Training:** [TRL `GRPOTrainer`](https://huggingface.co/docs/trl/en/grpo_trainer) with [Unsloth](https://github.com/unslothai/unsloth) 4-bit QLoRA on `Qwen/Qwen2.5-1.5B-Instruct`, using this repo's OpenEnv package (`cybersec/`). Outer loops collect on-policy rows then run GRPO steps — same pattern as [OpenEnv + TRL](https://huggingface.co/docs/trl/en/openenv). Step rewards use the environment default (`cybersec/reward.py`).

**Headless:** From the repo root, `python scripts/train_cybersec_grpo.py --output-dir ./_artifacts` (GPU + Unsloth + TRL; see root README).

**Config:** Tune the **`MODE`** dict in §1 for wall time vs. depth (`n_baseline_episodes`, `grpo_steps_per_outer_loop`, …). **`strict_canary`** gates post-training asserts.

**Artifacts:** Colab → **`WORKDIR`** (default `/content/cybersec_workdir`). Local → **`_artifacts/`** (under the process cwd).



## 0. Runtime, workdir, install (HF Space)

On Colab/Kaggle, artifacts and the package snapshot live under `WORKDIR`. 
The next cells mirror the iter notebooks: **`os.chdir(WORKDIR)`** and **`cybersec_env_pkg` → `cybersec`**.


In [ ]:
# Detect Colab / Kaggle / local. Artifacts + HF snapshot use WORKDIR on cloud kernels.
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    RUNTIME = "colab"
    WORKDIR = Path("/content/cybersec_workdir")
elif Path("/kaggle/working").exists() or os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    RUNTIME = "kaggle"
    WORKDIR = Path("/kaggle/working/cybersec_workdir")
else:
    RUNTIME = "local"
    WORKDIR = Path.cwd()

WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"runtime: {RUNTIME}")
print(f"workdir: {WORKDIR}")


In [ ]:
%%capture
# Pull env from HF Space into WORKDIR, normalize folder name, editable install.
# Renaming cybersec_env_pkg -> cybersec before pip install -e keeps the path stable.
import os, sys, subprocess
from pathlib import Path

HF_SPACE_REPO_ID = "Lonelyguyse1/cybersec"
HF_SPACE_REVISION = "main"

if RUNTIME in {"colab", "kaggle"}:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24", "openenv-core>=0.2.2"],
        check=True,
    )
    from huggingface_hub import snapshot_download

    PKG_STAGE = WORKDIR / "cybersec_env_pkg"
    CYBERSEC_SRC = WORKDIR / "cybersec"
    PKG_STAGE.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=HF_SPACE_REPO_ID,
        repo_type="space",
        revision=HF_SPACE_REVISION,
        local_dir=str(PKG_STAGE),
        local_dir_use_symlinks=False,
    )
    if CYBERSEC_SRC.exists():
        install_root = CYBERSEC_SRC
    elif PKG_STAGE.exists():
        PKG_STAGE.rename(CYBERSEC_SRC)
        install_root = CYBERSEC_SRC
    else:
        raise RuntimeError("snapshot_download did not populate", PKG_STAGE)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(install_root)], check=True)
    # Space snapshots can lag GitHub; this notebook imports training helpers (e.g.
    # collect_grpo_rows_from_rollouts) that may not be on the Space yet.
    _gh = "git+https://github.com/LonelyGuy-SE1/cybersec_env.git@main#subdirectory=cybersec"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", _gh],
        check=True,
    )
else:
    install_root = Path("..").resolve() / "cybersec"
    print(f"local runtime; using in-tree package at {install_root}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "unsloth[cu121] @ git+https://github.com/unslothai/unsloth.git"],
    check=False,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "trl", "peft", "accelerate", "bitsandbytes"],
    check=False,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "pandas"], check=False)


In [ ]:
# Legacy Colab/Kaggle: same as `%cd cybersec_workdir` then `os.rename('cybersec_env_pkg','cybersec')`.
# If install already renamed, this is a no-op.
import os
from pathlib import Path

if RUNTIME in {"colab", "kaggle"}:
    # Same effect as Colab `%cd /content/cybersec_workdir` (path is `WORKDIR`).
    _ipy = globals().get("get_ipython", lambda: None)()
    if _ipy is not None:
        _ipy.run_line_magic("cd", str(WORKDIR))
    else:
        os.chdir(WORKDIR)
    print("cwd:", Path.cwd())
    if Path("cybersec_env_pkg").exists() and not Path("cybersec").exists():
        os.rename("cybersec_env_pkg", "cybersec")
        print("renamed cybersec_env_pkg -> cybersec")


## 1. Imports & config


In [ ]:
from __future__ import annotations

# Iter-1's notebook output was buried under thousands of lines of identical
# `max_new_tokens / attention_mask` deprecation warnings. Silence the known
# noise *before* importing torch / transformers so the cell outputs are
# actually readable.
import warnings, logging
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("accelerate").setLevel(logging.ERROR)
logging.getLogger("peft").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("trl").setLevel(logging.WARNING)
logging.getLogger("websockets").setLevel(logging.WARNING)

import base64
import copy
import gc
import json
import math
import os
import pickle
import random
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg") if "google.colab" not in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset

from cybersec import (
    ActionType,
    CybersecAction,
    CybersecEnv,
    CybersecEnvironment,
    list_eval_scenarios,
    list_scenarios,
    list_train_scenarios,
)
from cybersec.baselines import (
    HeuristicPolicy,
    RandomPolicy,
    aggregate_results,
    run_episode,
)
from cybersec.training.rewards import (
    SYSTEM_PROMPT,
    collect_grpo_rows_from_rollouts,
    default_reward_funcs,
    parse_first_json_object,
    parsed_action,
    render_observation,
    restore_env,
    snapshot_env,
)

# ---------------------------------------------------------------------------
# Production MODE — tune for GPU memory and wall-clock.
# ---------------------------------------------------------------------------
MODE = {
    "strict_canary":           True,
    "n_baseline_episodes":     30,
    "n_dataset_seeds":         30,
    "max_dataset_rows":        1500,
    "on_policy_outer_loops":   3,
    "warmup_heuristic_outer0": True,
    "grpo_steps_per_outer_loop": 100,
    "grpo_num_train_epochs":   1,
    "grpo_num_generations":    6,
    "grpo_per_device_bs":      1,
    "grpo_grad_accum":         6,
    "grpo_logging_steps":      5,
    "grpo_save_steps":         50,
    "grpo_temperature":        1.2,
    "grpo_beta":               0.04,
    "grpo_learning_rate":      3e-6,
    "n_post_train_episodes":   30,
    "run_base_llm_eval":       True,
    "rollout_do_sample":       True,
    "rollout_temperature":     0.95,
    "rollout_top_p":           0.92,
    "eval_do_sample":          True,
    "eval_temperature":        0.85,
    "eval_top_p":              0.95,
    "use_remote_env":          False,
    "remote_env_url":          "https://lonelyguyse1-cybersec.hf.space",
}

MODE["grpo_max_steps_effective"] = (
    int(MODE["grpo_steps_per_outer_loop"]) * int(MODE["on_policy_outer_loops"])
)

if RUNTIME in ("colab", "kaggle"):
    ARTIFACTS = WORKDIR
else:
    ARTIFACTS = (Path.cwd() / "_artifacts").resolve()
ARTIFACTS.mkdir(parents=True, exist_ok=True)

ADAPTER_DIR        = ARTIFACTS / "qwen_cybersec_lora"
TRAIN_LOG          = ARTIFACTS / "training_log.json"
BASELINE_JSON      = ARTIFACTS / "baseline_metrics.json"
POST_JSON          = ARTIFACTS / "post_train_metrics.json"
HELDOUT_JSON       = ARTIFACTS / "heldout_metrics.json"
BASELINE_PLOT      = ARTIFACTS / "baseline_curves.png"
BEFORE_AFTER       = ARTIFACTS / "before_after_curves.png"
TRAIN_DIAGNOSTICS  = ARTIFACTS / "training_diagnostics.png"
SUMMARY_MD         = ARTIFACTS / "summary_table.md"
TRAJECTORY_JSON    = ARTIFACTS / "trajectory_dataset.jsonl"

MODEL_NAME      = "Qwen/Qwen2.5-1.5B-Instruct"
# GRPO applies left-truncation to `max_prompt_length` tokens; chat templates add overhead.
# Unsloth `max_seq_length` must be >= prompt + completion + template slack (TRL #3569).
MAX_PROMPT_LEN  = 2048
MAX_NEW_TOKENS  = 64
MAX_SEQ_LEN     = MAX_PROMPT_LEN + MAX_NEW_TOKENS + 256

TRAIN_SCENARIOS = list_train_scenarios()
HELDOUT_SCENARIOS = list_eval_scenarios()
ALL_SCENARIOS = list_scenarios()
SEEDS_BASELINE = list(range(MODE["n_baseline_episodes"]))
SEEDS_DATASET  = list(range(MODE["n_dataset_seeds"]))
SEEDS_POST     = list(range(MODE["n_post_train_episodes"]))

print("train scenarios:   ", TRAIN_SCENARIOS)
print("held-out scenarios:", HELDOUT_SCENARIOS)
print("MODE:")
print(json.dumps(MODE, indent=2))
if not os.environ.get("HF_TOKEN") and not os.environ.get("HUGGINGFACE_HUB_TOKEN"):
    print("Tip: set HF_TOKEN or HUGGINGFACE_HUB_TOKEN (e.g. Colab secrets) to avoid hub auth warnings when pulling models.")

# --- run_manifest.json: torch/CUDA + effective batch; extend after dataset & training ---
_eff_bs = MODE["grpo_per_device_bs"] * MODE["grpo_grad_accum"]
RUN_MANIFEST = ARTIFACTS / "run_manifest.json"
_manifest = {
    "started_at_utc":        time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "mode":                  MODE,
    "model_name":            MODEL_NAME,
    "max_prompt_len":        MAX_PROMPT_LEN,
    "max_new_tokens":        MAX_NEW_TOKENS,
    "torch_version":         torch.__version__,
    "cuda_available":        torch.cuda.is_available(),
    "cuda_version":          getattr(torch.version, "cuda", None),
    "device_name":           (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
    "effective_optim_batch":  _eff_bs,
    "notes":                 "GRPO uses fp16 on GPU (bf16 False); 4-bit base via Unsloth.",
    "bf16":                  False,
    "fp16":                  bool(torch.cuda.is_available()),
}
RUN_MANIFEST.write_text(json.dumps(_manifest, indent=2, default=str))
print("--- run_manifest (partial) ---")
print("  device:", _manifest["device_name"], "| fp16:", _manifest["fp16"], "| eff. batch (acc):", _eff_bs)
print("wrote", RUN_MANIFEST)


## 2. Baseline: Random vs Heuristic (train + held-out scenarios)


In [ ]:
# Local CybersecEnvironment for fast in-kernel rollouts. We optionally
# repeat the baseline against the deployed HF Space below as a separate
# optional remote OpenEnv check (§2a).
env = CybersecEnvironment()
baseline_runs = {}

t0 = time.time()
for sid in ALL_SCENARIOS:
    rand = [run_episode(env, RandomPolicy(seed=s), seed=s, scenario_id=sid) for s in SEEDS_BASELINE]
    heur = [run_episode(env, HeuristicPolicy(),    seed=s, scenario_id=sid) for s in SEEDS_BASELINE]
    baseline_runs[sid] = {"random": rand, "heuristic": heur}
    print(f"{sid:<32s}  random={aggregate_results(rand)['mean_return']:7.3f}"
          f"  heuristic={aggregate_results(heur)['mean_return']:7.3f}")
print(f"baseline elapsed: {time.time()-t0:.1f}s")


### 2a. (Optional) Remote environment check

If `MODE['use_remote_env']` is `True`, this cell runs **one** baseline
episode of the heuristic policy against the deployed HF Space env. The
purpose is to prove that the same package code, served over the OpenEnv
WebSocket protocol, returns identical-shape observations -- not to repeat
the full baseline against the network (which would be slow).


In [ ]:
import asyncio

async def _remote_env_probe():
    base_url = MODE["remote_env_url"]
    print(f"connecting to remote env: {base_url}")
    async with CybersecEnv(base_url=base_url) as remote:
        result = await remote.reset(seed=0, scenario_id=TRAIN_SCENARIOS[0])
        steps = 0
        cumulative = 0.0
        while not result.done and steps < 80:
            policy = HeuristicPolicy()
            action = policy.act(result.observation)
            result = await remote.step(action)
            cumulative += float(result.reward or 0.0)
            steps += 1
        print(f"remote heuristic episode: steps={steps}  cumulative_reward={cumulative:.3f}")

if MODE["use_remote_env"]:
    try:
        asyncio.run(_remote_env_probe())
    except Exception as exc:
        print(f"remote environment check failed (continuing with local): {exc!r}")
else:
    print("remote environment check skipped (MODE['use_remote_env'] is False)")


### 2b. Persist baseline metrics + plot


In [ ]:
def _padded_cumulative(curves, target_len):
    out = np.zeros((len(curves), target_len), dtype=float)
    for i, c in enumerate(curves):
        out[i, : len(c)] = c
        if len(c) < target_len:
            out[i, len(c):] = c[-1] if c else 0.0
    return np.cumsum(out, axis=1)

n_panels = len(ALL_SCENARIOS)
fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 4), sharey=True)
if n_panels == 1:
    axes = [axes]
for ax, sid in zip(axes, ALL_SCENARIOS):
    cell = baseline_runs[sid]
    horizon = max(max(len(r.reward_curve) for r in cell['random']),
                  max(len(r.reward_curve) for r in cell['heuristic']))
    for label, color in [("random", "tab:gray"), ("heuristic", "tab:blue")]:
        cumr = _padded_cumulative([r.reward_curve for r in cell[label]], horizon)
        mean = cumr.mean(axis=0); std = cumr.std(axis=0)
        ax.plot(mean, label=label, color=color)
        ax.fill_between(np.arange(horizon), mean - std, mean + std, color=color, alpha=0.15)
    held_out_tag = "  (HELD-OUT)" if sid in HELDOUT_SCENARIOS else ""
    ax.set_title(sid + held_out_tag, fontsize=10); ax.set_xlabel("tick"); ax.axhline(0, color="k", lw=0.5)
axes[0].set_ylabel("cumulative reward"); axes[0].legend()
fig.suptitle(f"Cybersec OpenEnv baselines, n={MODE['n_baseline_episodes']}/scenario")
fig.tight_layout(); fig.savefig(BASELINE_PLOT, dpi=140); plt.show()

baseline_metrics = {
    sid: {p: aggregate_results(cell[p]) for p in ("random", "heuristic")}
    for sid, cell in baseline_runs.items()
}
baseline_metrics["_meta"] = {
    "n_episodes": MODE["n_baseline_episodes"],
    "train_scenarios": TRAIN_SCENARIOS,
    "heldout_scenarios": HELDOUT_SCENARIOS,
}
BASELINE_JSON.write_text(json.dumps(baseline_metrics, indent=2))
print("wrote", BASELINE_JSON)


### 2c. Full-horizon episode trace (heuristic, illustration)

One episode: each row is a tick — **action** the policy took and a **short slice** of the text observation the model sees (from `render_observation`). This does not use the LLM; it shows the protocol. Trained-LLM trace is in §7c after eval.


In [ ]:
def run_and_print_episode_trace(env, policy, seed: int, scenario_id: str, obs_preview: int = 220):
    """Print tick-by-tick: what the policy saw, action, step reward, cumulative."""
    if hasattr(policy, "reset"):
        policy.reset()
    obs = env.reset(seed=seed, scenario_id=scenario_id)
    cumulative = 0.0
    rows = []
    while not obs.done:
        seen = render_observation(obs)[:obs_preview].replace("\n", " ")
        action = policy.act(obs)
        nxt = env.step(action)
        step_r = float(nxt.reward or 0.0)
        cumulative += step_r
        rows.append({
            "tick_after": nxt.tick,
            "action": action.model_dump(),
            "step_reward": round(step_r, 4),
            "cumulative": round(cumulative, 4),
            "observation_preview": seen,
        })
        obs = nxt
    tr = pd.DataFrame(rows)
    print(tr.to_string(index=False))
    return tr

# `env` is created in the baseline cell above; reuse it here.
_ = run_and_print_episode_trace(env, HeuristicPolicy(), seed=0, scenario_id=TRAIN_SCENARIOS[0])

## 3. Reward functions (eight independent signals)

All reward functions live in `cybersec.training.rewards` so the notebook,
the tests, and any future training script use the same definitions.


In [ ]:
REWARD_FUNCS = default_reward_funcs()
REWARD_NAMES = [f.__name__ for f in REWARD_FUNCS]
print("reward components:")
for n in REWARD_NAMES:
    print(f"  - {n}")


## 4. Load Qwen2.5-1.5B with Unsloth (4-bit QLoRA)


In [ ]:
from unsloth import FastLanguageModel


def _drop_gen_max_length(lm) -> None:
    """Causal LMs often ship `generation_config.max_length` (e.g. 32768). Calling
    `.generate(..., max_new_tokens=...)` merges that in and Transformers warns that
    both are set. Unset `max_length` so only `max_new_tokens` applies."""
    gen_cfg = getattr(lm, "generation_config", None)
    if gen_cfg is not None and getattr(gen_cfg, "max_length", None) is not None:
        gen_cfg.max_length = None


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=0,
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
_drop_gen_max_length(model)
model.print_trainable_parameters()

def to_chat_prompt(row):
    msgs = [
        {"role": "system", "content": row["system"]},
        {"role": "user",   "content": row["prompt"]},
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return {"prompt": text}


## 5. Outer loop: collect rows → GRPO

Each iteration builds a fresh `Dataset` via `collect_grpo_rows_from_rollouts` (`cybersec.training.rewards`): **loop 0** uses **HeuristicPolicy** when `warmup_heuristic_outer0` is true; later loops use the **current LLM** (on-policy rollouts). **Training scenarios only**; held-out scenarios are for evaluation. Rows are mapped with `to_chat_prompt` from §4 before `GRPOTrainer`.



In [ ]:
from trl import GRPOConfig, GRPOTrainer


class RolloutLLMPolicy:
    """Rollout policy using the *training* model weights (updates each outer loop)."""

    name = "rollout-llm"

    def __init__(self, lm, tok, *, do_sample: bool, temperature: float, top_p: float):
        self._lm = lm
        self._tok = tok
        self._do_sample = do_sample
        self._temperature = temperature
        self._top_p = top_p

    def reset(self) -> None:
        pass

    def act(self, obs):
        self._lm.eval()
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": render_observation(obs)},
        ]
        text = self._tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = self._tok(text, return_tensors="pt").to(self._lm.device)
        gen_kw = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=self._tok.eos_token_id,
        )
        if self._do_sample:
            gen_kw["do_sample"] = True
            gen_kw["temperature"] = self._temperature
            gen_kw["top_p"] = self._top_p
        else:
            gen_kw["do_sample"] = False
        with torch.inference_mode():
            out = self._lm.generate(**inputs, **gen_kw)
        completion = self._tok.decode(
            out[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
        )
        a = parsed_action(completion)
        return a or CybersecAction(action_type=ActionType.INVESTIGATE, target="INVALID_TARGET_SYNTAX")


_use_fp16 = bool(torch.cuda.is_available())
_combined_history: list = []
_global_step_offset: int = 0
_grpo_total_sec = 0.0
_outer_n = int(MODE["on_policy_outer_loops"])

for outer_i in range(_outer_n):
    use_h = outer_i == 0 and MODE.get("warmup_heuristic_outer0", True)
    tag = "heuristic_warmup" if use_h else "llm_on_policy"
    print(f"=== outer {outer_i + 1}/{_outer_n}  collect: {tag} ===")
    _t_collect0 = time.time()
    if use_h:
        _pol = HeuristicPolicy()
    else:
        _pol = RolloutLLMPolicy(
            model,
            tokenizer,
            do_sample=bool(MODE.get("rollout_do_sample", True)),
            temperature=float(MODE.get("rollout_temperature", 0.9)),
            top_p=float(MODE.get("rollout_top_p", 0.92)),
        )
    rows = collect_grpo_rows_from_rollouts(
        TRAIN_SCENARIOS,
        SEEDS_DATASET,
        _pol,
        max_rows=int(MODE["max_dataset_rows"]),
        shuffle_seed=outer_i + 7,
    )
    print(f"  collected {len(rows)} rows in {time.time() - _t_collect0:.1f}s")
    ds = Dataset.from_list(rows)
    ds_chat = ds.map(to_chat_prompt)
    print("  first chat prompt (trunc):", ds_chat[0]["prompt"][:400])

    _cfg = GRPOConfig(
        output_dir=str(ARTIFACTS / f"grpo_checkpoints_outer{outer_i}"),
        learning_rate=MODE["grpo_learning_rate"],
        per_device_train_batch_size=MODE["grpo_per_device_bs"],
        gradient_accumulation_steps=MODE["grpo_grad_accum"],
        max_prompt_length=MAX_PROMPT_LEN,
        max_completion_length=MAX_NEW_TOKENS,
        num_generations=MODE["grpo_num_generations"],
        num_train_epochs=MODE.get("grpo_num_train_epochs", 1),
        max_steps=int(MODE["grpo_steps_per_outer_loop"]),
        logging_steps=MODE["grpo_logging_steps"],
        save_steps=MODE["grpo_save_steps"],
        save_total_limit=1,
        bf16=False,
        fp16=_use_fp16,
        report_to=[],
        seed=outer_i,
        temperature=MODE["grpo_temperature"],
        beta=MODE["grpo_beta"],
    )
    print("  GRPO max_steps (this outer):", _cfg.max_steps, "num_generations:", _cfg.num_generations)

    trainer = GRPOTrainer(
        model=model,
        args=_cfg,
        train_dataset=ds_chat,
        processing_class=tokenizer,
        reward_funcs=REWARD_FUNCS,
    )
    _t1 = time.time()
    trainer.train()
    _loop_sec = time.time() - _t1
    _grpo_total_sec += _loop_sec
    _raw_hist = getattr(trainer.state, "log_history", []) or []
    for _row in _raw_hist:
        _r = dict(_row)
        _s = _r.get("step")
        if _s is not None:
            _r["global_step"] = _global_step_offset + int(_s)
        _r["outer_loop"] = outer_i
        _combined_history.append(_r)
    _global_step_offset += int(getattr(trainer.state, "global_step", 0) or 0)
    print(f"  GRPO outer {outer_i} elapsed: {_loop_sec:.1f}s")
    del trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"=== all outers GRPO time: {_grpo_total_sec:.1f}s ===")
if RUN_MANIFEST.exists():
    _mf = json.loads(RUN_MANIFEST.read_text())
    _mf["grpo_train_elapsed_s"] = round(_grpo_total_sec, 2)
    _mf["on_policy_outer_loops"] = _outer_n
    _mf["training_phase_finished_utc"] = time.strftime(
        "%Y-%m-%dT%H:%M:%SZ", time.gmtime()
    )
    _mf["dataset_rows_last_outer"] = len(rows)
    RUN_MANIFEST.write_text(json.dumps(_mf, indent=2, default=str))


## 6. Save adapter + training log


In [ ]:
ADAPTER_DIR.mkdir(exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

history = _combined_history
TRAIN_LOG.write_text(json.dumps({
    "reward_names": REWARD_NAMES,
    "history": history,
    "model_name": MODEL_NAME,
    "mode": MODE,
}, indent=2))
print("saved adapter to", ADAPTER_DIR)
print("wrote", TRAIN_LOG, "with", len(history), "log rows")
if RUN_MANIFEST.exists():
    _mf = json.loads(RUN_MANIFEST.read_text())
    _mf["training_log_path"] = str(TRAIN_LOG)
    _mf["training_log_rows"] = len(history)
    if history and isinstance(history[-1], dict):
        _mf["last_log_step"] = history[-1].get("global_step", history[-1].get("step"))
    RUN_MANIFEST.write_text(json.dumps(_mf, indent=2, default=str))


### 6a. Training diagnostics: KL, loss, per-component reward

Iter-1 had no KL/entropy plots, so we couldn't see the policy collapsing
toward a single canned plan even though the cumulative reward kept
climbing. These plots are the early-warning system: if KL is rising fast
and `reward_action_diversity` plateaus near `1/num_generations`, the
policy is mode-collapsing and you should stop and adjust.


In [ ]:
log = json.loads(TRAIN_LOG.read_text()) if TRAIN_LOG.exists() else {"history": []}
hist = log.get("history") or []

if not hist:
    print("no training log rows -- did GRPO train?")
else:
    df_log = pd.DataFrame(hist)
    _xcol = "global_step" if "global_step" in df_log.columns else "step"
    if _xcol not in df_log.columns:
        df_log[_xcol] = range(len(df_log))
        _xcol = "step"

    base_cols = ["loss", "kl", "reward", "completion_length", "grad_norm"]
    component_cols = [c for c in df_log.columns
                      if c.startswith("rewards/") or c in REWARD_NAMES
                      or any(c.endswith(name) for name in REWARD_NAMES)]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # panel 1: loss + kl
    for col, color in [("loss", "tab:blue"), ("kl", "tab:red")]:
        if col in df_log.columns and df_log[col].notna().any():
            axes[0].plot(df_log[_xcol], df_log[col], label=col, color=color)
    axes[0].set_title("loss / KL"); axes[0].set_xlabel(_xcol); axes[0].legend()

    # panel 2: total reward + completion length
    if "reward" in df_log.columns and df_log["reward"].notna().any():
        axes[1].plot(df_log[_xcol], df_log["reward"], label="reward", color="tab:green")
    if "completion_length" in df_log.columns and df_log["completion_length"].notna().any():
        ax2 = axes[1].twinx()
        ax2.plot(df_log[_xcol], df_log["completion_length"], label="completion_length",
                 color="tab:purple", linestyle="--")
        ax2.set_ylabel("completion_length", color="tab:purple")
    axes[1].set_title("total reward + completion length"); axes[1].set_xlabel(_xcol)
    axes[1].legend(loc="upper left")

    # panel 3: per-reward-component breakdown
    if component_cols:
        for col in component_cols:
            if df_log[col].notna().any():
                short = col.split("/")[-1].replace("reward_", "")
                axes[2].plot(df_log[_xcol], df_log[col], label=short, alpha=0.85)
        axes[2].set_title("reward components per step")
        axes[2].set_xlabel(_xcol)
        axes[2].legend(fontsize=7, ncol=2, loc="best")
    else:
        axes[2].set_title("(no per-component reward columns in log)")

    fig.tight_layout(); fig.savefig(TRAIN_DIAGNOSTICS, dpi=140); plt.show()
    print("wrote", TRAIN_DIAGNOSTICS)


## 7. Evaluate base LLM vs trained policy

1. **Base** `Qwen2.5-1.5B-Instruct` (no LoRA, same 4-bit load) on **all** scenarios
   — set `MODE["run_base_llm_eval"] = False` to skip and save time.
2. **Trained adapter** — reload through **Unsloth** so LoRA + fused attention match training.

Use `FastLanguageModel.from_pretrained` on `ADAPTER_DIR` after freeing VRAM from the base run.


In [ ]:
from unsloth import FastLanguageModel

# Free training-only buffers if the kernel still has them.
for _name in ("model", "trainer"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

base_runs: dict = {}
if MODE.get("run_base_llm_eval", True):
    base_eval_model, base_tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )
    base_tokenizer.pad_token = base_tokenizer.pad_token or base_tokenizer.eos_token
    FastLanguageModel.for_inference(base_eval_model)
    _drop_gen_max_length(base_eval_model)

    @torch.inference_mode()
    def _base_llm_act(obs):
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": render_observation(obs)},
        ]
        text = base_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = base_tokenizer(text, return_tensors="pt").to(base_eval_model.device)
        out = base_eval_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=base_tokenizer.eos_token_id,
        )
        completion = base_tokenizer.decode(
            out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
        )
        a = parsed_action(completion)
        _base_llm_act._last_was_fallback = a is None
        return a or CybersecAction(action_type=ActionType.INVESTIGATE, target="INVALID_TARGET_SYNTAX")

    class BaseLLMPolicy:
        name = "qwen-1.5b-base"
        last_act_was_fallback = False
        def reset(self):
            self.last_act_was_fallback = False
        def act(self, obs):
            act_out = _base_llm_act(obs)
            self.last_act_was_fallback = bool(getattr(_base_llm_act, "_last_was_fallback", False))
            return act_out

    t0 = time.time()
    for sid in ALL_SCENARIOS:
        runs = [run_episode(env, BaseLLMPolicy(), seed=s, scenario_id=sid) for s in SEEDS_POST]
        base_runs[sid] = runs
        agg = aggregate_results(runs)
        std = float(np.std([r.cumulative_reward for r in runs]))
        print(f"[base-llm] {sid:<32s}  mean={agg['mean_return']:7.3f}  std={std:.3f}  "
              f"mon_fb={agg.get('monitor_fallback_rate', 0):.3f}")
    print(f"base-llm eval elapsed: {time.time()-t0:.1f}s")

    del base_eval_model, base_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("skipping base-LLM eval (MODE['run_base_llm_eval'] is False)")

eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_DIR),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
eval_tokenizer.pad_token = eval_tokenizer.pad_token or eval_tokenizer.eos_token
FastLanguageModel.for_inference(eval_model)
_drop_gen_max_length(eval_model)
print("loaded trained adapter via Unsloth for evaluation")


In [ ]:
@torch.inference_mode()
def llm_act(obs):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": render_observation(obs)},
    ]
    text = eval_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = eval_tokenizer(text, return_tensors="pt").to(eval_model.device)
    out = eval_model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=eval_tokenizer.eos_token_id,
    )
    completion = eval_tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    a = parsed_action(completion)
    llm_act._last_was_fallback = a is None
    return a or CybersecAction(action_type=ActionType.INVESTIGATE, target="INVALID_TARGET_SYNTAX")

class TrainedLLMPolicy:
    name = "qwen-1.5b-grpo"
    last_act_was_fallback = False
    def reset(self):
        self.last_act_was_fallback = False
    def act(self, obs):
        out = llm_act(obs)
        self.last_act_was_fallback = bool(getattr(llm_act, "_last_was_fallback", False))
        return out

trained_runs = {}
t0 = time.time()
for sid in TRAIN_SCENARIOS:
    runs = [run_episode(env, TrainedLLMPolicy(), seed=s, scenario_id=sid) for s in SEEDS_POST]
    trained_runs[sid] = runs
    agg = aggregate_results(runs)
    print(f"{sid:<32s}  trained-llm={agg['mean_return']:7.3f}  "
          f"std={float(np.std([r.cumulative_reward for r in runs])):.3f}  "
          f"mon_fb={agg.get('monitor_fallback_rate', 0):.3f}")
print(f"trained-policy eval (train scenarios) elapsed: {time.time()-t0:.1f}s")


### 7a. Held-out OOD evaluation

The whole point of holding out `cloud_metadata_ssrf` from training is
this cell: a clean read on whether the trained policy *generalises* or
just memorised per-scenario plans.


In [ ]:
heldout_runs = {}
t0 = time.time()
for sid in HELDOUT_SCENARIOS:
    runs = [run_episode(env, TrainedLLMPolicy(), seed=s, scenario_id=sid) for s in SEEDS_POST]
    heldout_runs[sid] = runs
    agg = aggregate_results(runs)
    rand_mean = aggregate_results(baseline_runs[sid]['random'])['mean_return']
    heur_mean = aggregate_results(baseline_runs[sid]['heuristic'])['mean_return']
    std = float(np.std([r.cumulative_reward for r in runs]))
    print(f"{sid:<32s}  trained-llm={agg['mean_return']:7.3f}  std={std:.3f}  "
          f"vs heur={heur_mean:.3f}  vs rand={rand_mean:.3f}")
print(f"trained-policy eval (held-out scenarios) elapsed: {time.time()-t0:.1f}s")

heldout_metrics = {
    sid: {
        "random":      aggregate_results(baseline_runs[sid]["random"]),
        "heuristic":   aggregate_results(baseline_runs[sid]["heuristic"]),
        "trained_llm": aggregate_results(heldout_runs[sid]),
    }
    for sid in HELDOUT_SCENARIOS
}
HELDOUT_JSON.write_text(json.dumps(heldout_metrics, indent=2))
print("wrote", HELDOUT_JSON)


### 7c. Full-horizon trace (trained LLM)

Same table as §2c, but actions come from the **fine-tuned** policy (`TrainedLLMPolicy` + loaded adapter). Requires §7 cells to have run.


In [ ]:
print("--- trained LLM full-episode trace ---")
_ = run_and_print_episode_trace(
    env, TrainedLLMPolicy(), seed=0, scenario_id=TRAIN_SCENARIOS[0], obs_preview=200
)

## 8. Before/after curves on identical axes


In [ ]:
all_post = {**trained_runs, **heldout_runs}
panels = ALL_SCENARIOS
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4), sharey=True)
if len(panels) == 1:
    axes = [axes]
for ax, sid in zip(axes, panels):
    cell = baseline_runs[sid]
    llm  = all_post.get(sid, [])
    base_ep = (base_runs or {}).get(sid, [])
    horizon = max(
        max(len(r.reward_curve) for r in cell['random']),
        max(len(r.reward_curve) for r in cell['heuristic']),
        max((len(r.reward_curve) for r in llm), default=1),
        max((len(r.reward_curve) for r in base_ep), default=1),
    )
    palette = [("random", cell['random'], "tab:gray"),
               ("heuristic", cell['heuristic'], "tab:blue"),
               ("base-llm", base_ep, "tab:green"),
               ("trained-llm", llm, "tab:red")]
    for label, runs, color in palette:
        if not runs:
            continue
        if label == "base-llm" and not base_runs:
            continue
        cumr = _padded_cumulative([r.reward_curve for r in runs], horizon)
        mean = cumr.mean(axis=0); std = cumr.std(axis=0)
        ax.plot(mean, label=label, color=color)
        ax.fill_between(np.arange(horizon), mean - std, mean + std, color=color, alpha=0.15)
    held_out_tag = "  (HELD-OUT)" if sid in HELDOUT_SCENARIOS else ""
    ax.set_title(sid + held_out_tag, fontsize=10); ax.set_xlabel("tick"); ax.axhline(0, color="k", lw=0.5)
axes[0].set_ylabel("cumulative reward"); axes[0].legend()
fig.suptitle(
    f"Cybersec OpenEnv -- pre/post GRPO ({MODE['grpo_max_steps_effective']} opt steps), "
    f"n={MODE['n_post_train_episodes']}/scenario"
)
fig.tight_layout(); fig.savefig(BEFORE_AFTER, dpi=140); plt.show()


## 9. Summary table + headline delta


In [ ]:
rows = []
for sid in ALL_SCENARIOS:
    cell = baseline_runs[sid]
    post_runs = all_post.get(sid, [])
    br = (base_runs or {}).get(sid, [])
    policy_rows = [
        ("random",      cell["random"]),
        ("heuristic",   cell["heuristic"]),
        ("base-llm",    br),
        ("trained-llm", post_runs),
    ]
    for policy_name, runs in policy_rows:
        if not runs:
            continue
        agg = aggregate_results(runs)
        returns = [r.cumulative_reward for r in runs]
        rows.append({
            "scenario":     sid,
            "split":        "held-out" if sid in HELDOUT_SCENARIOS else "train",
            "policy":       policy_name,
            "mean_return":  agg["mean_return"],
            "std_return":   round(float(np.std(returns)), 3) if returns else 0.0,
            "mean_stages":  agg["mean_stages_succeeded"],
            "exfil_rate":   agg["exfil_rate"],
            "invalid_rate": agg["mean_invalid_actions"],
            "mon_fallback":  agg.get("monitor_fallback_rate", 0.0),
        })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

post_metrics = {
    sid: {
        "random":      aggregate_results(baseline_runs[sid]["random"]),
        "heuristic":   aggregate_results(baseline_runs[sid]["heuristic"]),
        "base_llm":     aggregate_results((base_runs or {}).get(sid, [])) if (base_runs or {}).get(sid) else None,
        "trained_llm": aggregate_results(all_post.get(sid, [])) if all_post.get(sid) else None,
    }
    for sid in ALL_SCENARIOS
}
post_metrics["_meta"] = {
    "n_post_episodes": MODE["n_post_train_episodes"],
    "grpo_max_steps_effective": MODE["grpo_max_steps_effective"],
    "grpo_steps_per_outer_loop": MODE["grpo_steps_per_outer_loop"],
    "on_policy_outer_loops": MODE["on_policy_outer_loops"],
    "grpo_num_train_epochs": MODE.get("grpo_num_train_epochs", 1),
    "grpo_precision":    "fp16" if torch.cuda.is_available() else "fp32",
    "run_manifest":    str(RUN_MANIFEST),
    "model":           MODEL_NAME,
    "adapter":         str(ADAPTER_DIR),
    "run_base_llm_eval": bool(MODE.get("run_base_llm_eval", True)),
    "train_scenarios": TRAIN_SCENARIOS,
    "heldout_scenarios": HELDOUT_SCENARIOS,
    "reward_funcs":    REWARD_NAMES,
}
POST_JSON.write_text(json.dumps(post_metrics, indent=2))
SUMMARY_MD.write_text(df.to_markdown(index=False))
print("wrote", POST_JSON)
print("wrote", SUMMARY_MD)


In [ ]:
print("=== Headline delta (trained-llm vs heuristic) ===")
for sid in ALL_SCENARIOS:
    h = aggregate_results(baseline_runs[sid]["heuristic"])["mean_return"]
    runs = all_post.get(sid, [])
    if not runs:
        continue
    t = aggregate_results(runs)["mean_return"]
    tag = "(HELD-OUT)" if sid in HELDOUT_SCENARIOS else "(train)"
    print(f"{sid:<32s} {tag:<11s} heuristic={h:7.3f}  trained-llm={t:7.3f}  delta={t-h:+.3f}")


## 10. Sanity assertions (strict canaries)

When `MODE["strict_canary"]` is **True** (recommended for submission runs), this
cell **asserts** on: aggregate heuristic-vs-random, valid-action rate,
train-vs-random floor, **≥2/3 training scenarios** with `std_return > 0.1`
(anti-collapse), and a cap on **MONITOR parse-fallback** rate.
Set `strict_canary: False` in `MODE` only for local experimentation when you need prints without asserts.


In [ ]:
# 1. Reward shaping is healthy: heuristic must out-earn random *on aggregate*.
_sc = MODE.get("strict_canary", True)
heur_total = sum(
    float(np.mean([r.cumulative_reward for r in cell['heuristic']]))
    for cell in baseline_runs.values()
)
rand_total = sum(
    float(np.mean([r.cumulative_reward for r in cell['random']]))
    for cell in baseline_runs.values()
)
print(f"baseline totals: heuristic={heur_total:+.3f}  random={rand_total:+.3f}")
if _sc:
    assert heur_total > rand_total, (
        f"reward shaping regressed: heuristic total ({heur_total:.3f}) "
        f"<= random total ({rand_total:.3f}) summed across scenarios"
    )

# 2. Trained policy must produce mostly-valid actions across all trained
#    episodes. Aggregate over train + held-out to keep one number.
total_steps = sum(len(r.reward_curve) for runs in all_post.values() for r in runs)
total_invalid = sum(r.invalid_action_count for runs in all_post.values() for r in runs)
valid_rate = 1.0 - (total_invalid / max(1, total_steps))
print(f"valid-action rate across all post-train episodes: {valid_rate:.1%}  "
      f"({total_invalid} invalid / {total_steps} steps)")
if _sc:
    assert valid_rate >= 0.8, (
        f"trained policy is producing too many invalid actions ({valid_rate:.1%} valid)"
    )

# 3. Trained policy must not be catastrophically worse than random on any
#    *training* scenario. (Held-out scenarios are allowed to be worse --
#    that's the point of generalisation reporting; we just print it.)
for sid in TRAIN_SCENARIOS:
    rand = float(np.mean([r.cumulative_reward for r in baseline_runs[sid]['random']]))
    llm  = float(np.mean([r.cumulative_reward for r in trained_runs[sid]]))
    if _sc:
        assert llm >= rand - 5.0, (
            f"{sid}: trained-llm ({llm:.3f}) is more than 5 points below random "
            f"({rand:.3f}) -- training likely diverged"
        )

# 4. Strict reward-hack canary: std_return > 0.1 on at least (n_train - 1)
#    i.e. 2 of 3 training scenarios — catches iter-3/4 single-scenario variance.
non_zero_std_scenarios = []
for sid in TRAIN_SCENARIOS:
    returns = [r.cumulative_reward for r in trained_runs[sid]]
    s = float(np.std(returns))
    print(f"{sid:<32s}  trained-llm std_return={s:.4f}")
    if s > 0.1:
        non_zero_std_scenarios.append(sid)
_required = max(2, len(TRAIN_SCENARIOS) - 1)
if _sc:
    assert len(non_zero_std_scenarios) >= _required, (
        f"REWARD HACK CANARY: trained-llm has std_return < 0.1 on "
        f"{len(TRAIN_SCENARIOS) - len(non_zero_std_scenarios)} of {len(TRAIN_SCENARIOS)} "
        f"training scenarios (non-zero: {non_zero_std_scenarios}). "
        f"Try raising grpo_num_generations / grpo_temperature, grpo_beta, or grpo_steps_per_outer_loop."
    )

# 5. Parse / MONITOR fallback must stay bounded (unparseable JSON -> MONITOR).
total_fb = sum(r.monitor_fallback_count for runs in all_post.values() for r in runs)
fb_rate = total_fb / max(1, total_steps)
print(f"monitor_fallback rate (trained): {fb_rate:.1%}")
if _sc:
    assert fb_rate <= 0.5, (
        f"trained-llm falls back to MONITOR on {fb_rate:.1%} of steps — check completions"
    )

print("all sanity checks passed" if _sc else "strict_canary=False: diagnostics only (asserts skipped)")


## 11. (Optional) Push artifacts back to the HF Space

If you want the trained adapter and updated metrics to live on the HF
Space alongside the env (so the judges can diff `_artifacts/` between
iter-1 and iter-2), enable this cell. It uses the same
`huggingface_hub` API as the install cell.


In [ ]:
# Set this to True after a successful run if you want to upload artifacts.
PUSH_ARTIFACTS = False

if PUSH_ARTIFACTS:
    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_folder(
        folder_path=str(ARTIFACTS),
        path_in_repo="_artifacts",
        repo_id=HF_SPACE_REPO_ID,
        repo_type="space",
        commit_message="training artifacts",
    )
    print("uploaded _artifacts/ to", HF_SPACE_REPO_ID)
else:
    print("artifact push skipped (set PUSH_ARTIFACTS=True to enable)")
